<a href="https://colab.research.google.com/github/richirey75/Data-Mining-CS4990/blob/final/DecisionTree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
https://github.com/Suji04/ML_from_Scratch/blob/master/decision%20tree%20classification.ipynb

In [1]:
import numpy as np
import pandas as pd
from google.colab import drive

In [ ]:
# mount drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# move to our desire directories
%cd /content/drive/MyDrive/CS4990GroupProject/CSV_files

/content/drive/MyDrive/CS4990GroupProject/CSV_files


In [ ]:
# read our data and choose the columns we want using index
data = pd.read_csv("Oasis_tracks_info.csv", usecols=[8, 9, 17], skiprows=1, header=None)
data.head(10)

# data = data.drop(data.columns[[0, 1, 2, 3, 4, 5, 6, 7, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25]], axis=1)


,8,9,17
0,0.3150,0.970,0.4810
1,0.0839,0.929,0.1600
2,0.2860,0.822,0.1900
3,0.2560,0.908,0.3410
4,0.3140,0.824,0.0546
5,0.3590,0.913,0.5890
6,0.1150,0.945,0.1950
7,0.2940,0.742,0.4400
8,0.3050,0.988,0.0513
9,0.0657,0.943,0.2640


In [ ]:
class Node():
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, info_gain=None, value=None):
        ''' constructor '''

        # for decision node
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.info_gain = info_gain

        # for leaf node
        self.value = value

In [ ]:
class DecisionTreeClassifier():
    def __init__(self, min_samples_split=2, max_depth=2):
        ''' constructor '''

        # initialize the root of the tree
        self.root = None

        # stopping conditions
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth

    def build_tree(self, dataset, curr_depth=0):
        ''' recursive function to build the tree '''

        X, Y = dataset[:,:-1], dataset[:,-1]
        num_samples, num_features = np.shape(X)

        # split until stopping conditions are met
        if num_samples>=self.min_samples_split and curr_depth<=self.max_depth:
            # find the best split
            best_split = self.get_best_split(dataset, num_samples, num_features)
            # check if information gain is positive
            if best_split["info_gain"]>0:
                # recur left
                left_subtree = self.build_tree(best_split["dataset_left"], curr_depth+1)
                # recur right
                right_subtree = self.build_tree(best_split["dataset_right"], curr_depth+1)
                # return decision node
                return Node(best_split["feature_index"], best_split["threshold"],
                            left_subtree, right_subtree, best_split["info_gain"])

        # compute leaf node
        leaf_value = self.calculate_leaf_value(Y)
        # return leaf node
        return Node(value=leaf_value)

    def get_best_split(self, dataset, num_samples, num_features):
        ''' function to find the best split '''

        # dictionary to store the best split
        best_split = {}
        max_info_gain = -float("inf")

        # loop over all the features
        for feature_index in range(num_features):
            feature_values = dataset[:, feature_index]
            possible_thresholds = np.unique(feature_values)
            # loop over all the feature values present in the data
            for threshold in possible_thresholds:
                # get current split
                dataset_left, dataset_right = self.split(dataset, feature_index, threshold)
                # check if childs are not null
                if len(dataset_left)>0 and len(dataset_right)>0:
                    y, left_y, right_y = dataset[:, -1], dataset_left[:, -1], dataset_right[:, -1]
                    # compute information gain
                    curr_info_gain = self.information_gain(y, left_y, right_y, "gini")
                    # update the best split if needed
                    if curr_info_gain>max_info_gain:
                        best_split["feature_index"] = feature_index
                        best_split["threshold"] = threshold
                        best_split["dataset_left"] = dataset_left
                        best_split["dataset_right"] = dataset_right
                        best_split["info_gain"] = curr_info_gain
                        max_info_gain = curr_info_gain

        # return best split
        return best_split

    def split(self, dataset, feature_index, threshold):
        ''' function to split the data '''

        dataset_left = np.array([row for row in dataset if row[feature_index]<=threshold])
        dataset_right = np.array([row for row in dataset if row[feature_index]>threshold])
        return dataset_left, dataset_right

    def information_gain(self, parent, l_child, r_child, mode="entropy"):
        ''' function to compute information gain '''

        weight_l = len(l_child) / len(parent)
        weight_r = len(r_child) / len(parent)
        if mode=="gini":
            gain = self.gini_index(parent) - (weight_l*self.gini_index(l_child) + weight_r*self.gini_index(r_child))
        else:
            gain = self.entropy(parent) - (weight_l*self.entropy(l_child) + weight_r*self.entropy(r_child))
        return gain

    def entropy(self, y):
        ''' function to compute entropy '''

        class_labels = np.unique(y)
        entropy = 0
        for cls in class_labels:
            p_cls = len(y[y == cls]) / len(y)
            entropy += -p_cls * np.log2(p_cls)
        return entropy

    def gini_index(self, y):
        ''' function to compute gini index '''

        class_labels = np.unique(y)
        gini = 0
        for cls in class_labels:
            p_cls = len(y[y == cls]) / len(y)
            gini += p_cls**2
        return 1 - gini

    def calculate_leaf_value(self, Y):
        ''' function to compute leaf node '''

        Y = list(Y)
        return max(Y, key=Y.count)









    def print_tree(self, tree=None, indent=""):
      ''' Function to print the decision tree with better visualization '''

      if not tree:
          tree = self.root

      # If the current node is a leaf, print the value
      if tree.value is not None:
          print(f"{indent}Leaf: {tree.value}")
      else:
          # Print the feature being split on, the threshold, and the information gain
          print(f"{indent}Node: [Feature: {tree.feature_index} <= {tree.threshold}] (Info Gain: {tree.info_gain:.4f})")

          # Print left branch
          print(f"{indent}|--- Left:")
          self.print_tree(tree.left, indent + "    ")

          # Print right branch
          print(f"{indent}|--- Right:")
          self.print_tree(tree.right, indent + "    ")








    '''
    def print_tree(self, tree=None, indent=" "):
         function to print the tree

        if not tree:
            tree = self.root

        if tree.value is not None:
            print(tree.value)

        else:
            print("X_"+str(tree.feature_index), "<=", tree.threshold, "?", tree.info_gain)
            print("%sleft:" % (indent), end="")
            self.print_tree(tree.left, indent + indent)
            print("%sright:" % (indent), end="")
            self.print_tree(tree.right, indent + indent)
    '''





    def fit(self, X, Y):
        ''' function to train the tree '''

        dataset = np.concatenate((X, Y), axis=1)
        self.root = self.build_tree(dataset)

    def predict(self, X):
        ''' function to predict new dataset '''

        preditions = [self.make_prediction(x, self.root) for x in X]
        return preditions

    def make_prediction(self, x, tree):
        ''' function to predict a single data point '''

        if tree.value!=None: return tree.value
        feature_val = x[tree.feature_index]
        if feature_val<=tree.threshold:
            return self.make_prediction(x, tree.left)
        else:
            return self.make_prediction(x, tree.right)

In [ ]:
X = data.iloc[:, :-1].values
Y = data.iloc[:, -1].values.reshape(-1,1)
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=.2, random_state=41)

In [ ]:
classifier = DecisionTreeClassifier(min_samples_split=3, max_depth=3)
classifier.fit(X_train,Y_train)
classifier.print_tree()

Node: [Feature: 0 <= 0.0839] (Info Gain: 0.0082)
|--- Left:
    Node: [Feature: 0 <= 0.0657] (Info Gain: 0.4444)
    |--- Left:
        Leaf: 0.264
    |--- Right:
        Leaf: 0.16
|--- Right:
    Node: [Feature: 0 <= 0.567] (Info Gain: 0.0075)
    |--- Left:
        Node: [Feature: 0 <= 0.115] (Info Gain: 0.0073)
        |--- Left:
            Node: [Feature: 0 <= 0.113] (Info Gain: 0.3200)
            |--- Left:
                Leaf: 0.223
            |--- Right:
                Leaf: 0.195
        |--- Right:
            Node: [Feature: 1 <= 0.985] (Info Gain: 0.0072)
            |--- Left:
                Leaf: 0.622
            |--- Right:
                Leaf: 0.0513
    |--- Right:
        Node: [Feature: 0 <= 0.568] (Info Gain: 0.2292)
        |--- Left:
            Leaf: 0.585
        |--- Right:
            Node: [Feature: 1 <= 0.435] (Info Gain: 0.2778)
            |--- Left:
                Leaf: 0.305
            |--- Right:
                Leaf: 0.28


In [ ]:
classifier = DecisionTreeClassifier(min_samples_split=3, max_depth=3)
classifier.fit(X_train,Y_train)
classifier.print_tree()

X_0 <= 0.0839 ? 0.008232844778084503
 left:X_0 <= 0.0657 ? 0.4444444444444444
  left:0.264
  right:0.16
 right:X_0 <= 0.567 ? 0.007521294546610324
  left:X_0 <= 0.115 ? 0.007276198863865768
    left:X_0 <= 0.113 ? 0.31999999999999995
        left:0.223
        right:0.195
    right:X_1 <= 0.985 ? 0.007181947816703338
        left:0.622
        right:0.0513
  right:X_0 <= 0.568 ? 0.22916666666666674
    left:0.585
    right:X_1 <= 0.435 ? 0.2777777777777777
        left:0.305
        right:0.28


In [ ]:
Y_pred = classifier.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy_score(Y_test, Y_pred)

In [ ]:
def print_tree(self, tree=None, indent=""):
    ''' Function to print the decision tree with better visualization '''

    if not tree:
        tree = self.root

    # If the current node is a leaf, print the value
    if tree.value is not None:
        print(f"{indent}Leaf: {tree.value}")
    else:
        # Print the feature being split on, the threshold, and the information gain
        print(f"{indent}Node: [Feature: {tree.feature_index} <= {tree.threshold}] (Info Gain: {tree.info_gain:.4f})")

        # Print left branch
        print(f"{indent}|--- Left:")
        self.print_tree(tree.left, indent + "    ")

        # Print right branch
        print(f"{indent}|--- Right:")
        self.print_tree(tree.right, indent + "    ")

In [ ]:
def print_tree(self, tree=None, indent=" "):
        ''' function to print the tree '''

        if not tree:
            tree = self.root

        if tree.value is not None:
            print(tree.value)

        else:
            print("X_"+str(tree.feature_index), "<=", tree.threshold, "?", tree.info_gain)
            print("%sleft:" % (indent), end="")
            self.print_tree(tree.left, indent + indent)
            print("%sright:" % (indent), end="")
            self.print_tree(tree.right, indent + indent)

# Test cases

In [ ]:
import sys
import classification
import pandas
import numpy
import random
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder

# Naive Bayes using sklearn
class NaiveBayes:
    def fit(self, x,y):
        # NB needs the categories to be converted to numbers
        # The ordinal encoder does exactly that
        self.encoder = OrdinalEncoder()
        self.NB = CategoricalNB()
        # encode the data
        xenc = self.encoder.fit_transform(x)
        # fit the naive bayes model
        self.NB.fit(xenc,y)
    def predict(self, x):
        # to predict, we first need to encode the data (using the same encoder)
        # and then predict using the NB model
        return self.NB.predict(self.encoder.transform(x))
    def to_dict(self):
        # Just to have some representation for the classifier
        return {"encoder": str(self.encoder), "nb": str(self.NB)}

CATMAP = {"low": 0, "medium": 1, "high": 2}
BINMAP = {False: 0, True: 1}
COLORS = {0: "Red", 1: "Blue"}


def plot(title, xs, ys, predys, mapping=CATMAP):
    markers = dict(zip(set(ys), ["+", "_", "*", "^", "s", "D"]))
    x1s = {}
    x2s = {}
    colors = {}
    for m in markers:
        x1s[m] = []
        x2s[m] = []
        colors[m] = []
    for x,y,py in zip(xs,ys,predys):

        x1s[y].append(mapping[x[0]]+random.gauss(0,0.05))
        x2s[y].append(mapping[x[1]]+random.gauss(0,0.05))
        colors[y].append(COLORS[py])
    for m in markers:
        plt.scatter(x1s[m], x2s[m], c=colors[m], marker=markers[m])
    plt.show()

def evaluate(prefix, y, predy):
    correct = 0
    for p,v in zip(predy, y):
        if p == v: correct += 1
    print("%sAccuracy: %.2f"%(prefix, correct*1.0/len(y)))

def get_columns(rows, columns, single=False):
    if single:
        return [row[columns[0]] for row in rows]
    return [[row[c] for c in columns] for row in rows]

CLASSIFICATION_TESTS = ["Predict class from two categories (1+2)", "Predict class from two categories (3+4)", "Predict class from all four categories", "Predict class from three categories (2-4)", "Predict class from two binary attributes (1+2)", "Predict class from three binary attributes (3+4)", "Predict class from wrong categorical attributes", "Predict class from wrong binary attributes"]

MODELS = {"Decision Tree": classification.DecisionTree, "Naive Bayes": NaiveBayes}

def classification_testcase(training, validation, n, visualize=True, model="Decision Tree"):
    print("running test:", CLASSIFICATION_TESTS[n])
    if n == 0:
        columns = ["cat1", "cat2"]
        target = ["cls3"]
        mapping = CATMAP
    elif n == 1:
        columns = ["cat3", "cat4"]
        target = ["cls3"]
        mapping = CATMAP
    elif n == 2:
        columns = ["cat1", "cat2", "cat3", "cat4"]
        target = ["cls3"]
        visualize = False
    elif n == 3:
        columns = ["cat2", "cat3", "cat4"]
        target = ["cls3"]
        visualize = False
    elif n == 4:
        columns = ["bin1", "bin2"]
        target = ["cls4"]
        mapping = BINMAP
    elif n == 5:
        columns = ["bin3", "bin4", "bin5"]
        target = ["cls4"]
        visualize = False
    elif n == 6:
        columns = ["bin1", "bin2", "bin3", "bin4", "bin5"]
        target = ["cls4"]
        visualize = False
    elif n == 7:
        columns = ["cat1", "cat2"]
        target = ["cls4"]
        mapping = CATMAP
    elif n == 8:
        columns = ["bin1", "bin2", "bin3", "bin4", "bin5"]
        target = ["cls3"]
        visualize = False

    m = MODELS[model]()
    tx = get_columns(training, columns)
    ty = get_columns(training, target, single=True)
    m.fit(tx, ty)
    print(json.dumps(m.to_dict(), indent=4))
    predty = m.predict(tx)
    evaluate(model + " training ", ty, predty)
    vx = get_columns(validation, columns)
    vy = get_columns(validation, target, single=True)
    predvy = m.predict(vx)
    evaluate(model + " validation ", vy, predvy)

    if visualize:
        plot(model + " training set", tx, ty, predty, mapping)
        plot(model + " validation set", vx, vy, predvy, mapping)

def main(auto=False, nb=False, steps=[]):
    random.seed(1337)
    df = pandas.read_csv("testdata.csv")
    model = "Decision Tree"
    if nb:
        model = "Naive Bayes"
    training = []
    validation = []
    for i,row in df.iterrows():
        if random.random() > 0.85:
            validation.append(row)
        else:
            training.append(row)

    if auto:
        for i,t in enumerate(CLASSIFICATION_TESTS):
            print("-"*80)
            classification_testcase(training, validation, i, False, model)
        return
    tests = CLASSIFICATION_TESTS
    while True:
        print("Which test case do you want to run?")
        for i,t in enumerate(tests):
            print(f"   {i} {t}")
        print("   q exit")
        if steps:
            x = steps[0]
            del steps[0]
        else:
            x = input("> ")
        if x in [str(i) for i,_ in enumerate(tests)]:
            classification_testcase(training, validation, int(x), model=model)
        elif x == "q":
            print("Bye")
            sys.exit(0)
        else:
            print("Please select a test case, r or q")
        print()

if __name__ == "__main__":
    if "--help" in sys.argv:
        print("Usage: testcases.py [--auto] steps")
        print("   --auto runs the tests automatically")
        print("   --naive-bayes uses the Naive Bayes classifier from sklearn; useful as a comparison")
        print("   <steps> is a sequence of inputs that are passed to the menu before it accepts manual input.")
        print("           This allows you to run e.g. 'python testcases.py 12q' to run test cases 1 and 2")
        print("           in sequence, followed by q(uit)")
        print("           Essentially, this allows you to repeatedly run any test/combination of tests without")
        print("           having to navigate the menu every time.")
        sys.exit(0)
    else:
        main("--auto" in sys.argv, "--bayes" in sys.argv or "--naive-bayes" in sys.argv or "-n" in sys.argv,
             list("".join([arg for arg in sys.argv[1:] if "-" not in arg])))